# CS 3110/3990: Data Privacy
## In-Class Exercise #4, week of 9/28/2026

In [17]:
# Load the data and libraries
import pandas as pd
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt

def laplace_mech(v, sensitivity, epsilon):
    return v + np.random.laplace(loc=0, scale=sensitivity / epsilon)

def pct_error(orig, priv):
    return np.abs(orig - priv)/orig * 100.0

adult = pd.read_csv('https://github.com/jnear/cs3110-data-privacy/raw/main/homework/adult_with_pii.csv')

## Question 1

For various values of $b$, write code to print out the percent error of summing the ages in the `adult` dataset, 
clipped to each value of $b$.

In [18]:
bs = range(1, 100, 10)
real_sum = adult['Age'].sum()

for b in bs:
    clipped_sum = adult['Age'].clip(lower=0, upper = b).sum()
    print(f"% error for {b} is {pct_error(real_sum, clipped_sum)}")

% error for 1 is 97.4080940444511
% error for 11 is 71.48903448896206
% error for 21 is 46.000380495392264
% error for 31 is 25.509031989473492
% error for 41 is 11.7959939725709
% error for 51 is 4.329687317165198
% error for 61 is 1.1587597123836921
% error for 71 is 0.2341877497996031
% error for 81 is 0.03860675005194001
% error for 91 is 0.0


What value of $b$ is the best?

Any value of b > than 81 is fine, at b = 81 the relative error due to bias is small. Values much larger than this risk adding attitional error when we add noise to achieve DP. 

## Question 2

For various values of $b$, print the result of a *differentially private* sum of ages, clipped to each value of $b$. Use $\epsilon = 0.1$.

In [19]:
bs = range(1, 100, 10)
real_sum = adult['Age'].sum()

# If pct_error is removed, DP is satisfied
# For a total privacy cost of ep = 10 * 0.1 by sequential compostion
for b in bs:
    clipped_sum = adult['Age'].clip(lower=0, upper = b).sum()
    dp_sum=laplace_mech(clipped_sum, sensitivity=b, epsilon=0.1)
    print(f"fDP result for {b} is {dp_sum}, error is {pct_error(real_sum, dp_sum)}")

fDP result for 1 is 32570.841278528085, error is 97.40731066346073
fDP result for 11 is 358311.9070499712, error is 71.47781806987176
fDP result for 21 is 678349.2212320177, error is 46.00235292364399
fDP result for 31 is 935684.7275567771, error is 25.51804865112974
fDP result for 41 is 1106969.5673909585, error is 11.883510508521859
fDP result for 51 is 1201792.1100289784, error is 4.33548947158277
fDP result for 61 is 1242020.0350772738, error is 1.1332844252988166
fDP result for 71 is 1252097.0180064533, error is 0.3311410000936679
fDP result for 81 is 1255560.496261668, error is 0.055442774713460236
fDP result for 91 is 1255814.272809975, error is 0.03524176900307166


Which value of $b$ is the best now? Does it differ?

It does not really differ, 81 or 91 is still good, the error due to bias is much greater than the error due to variance, until the clipping param is really big

## Question 3

Write an algorithm to *automatically pick the clipping parameter* for the age column. Your algorithm should satisfy differential privacy.


Look for a bunch of canidates for b, like Q2
Find thte first pair(b, b') st noisy val for b > noisy val for b'

In [20]:
def pick_b(epsilon):
    bs = range(1, 200, 20)
    real_sum = adult['Age'].sum()
    epsilon_i = epsilon / len(bs)
    dp_sums = []
    last_sum = 0
    for b in bs: #Satisfies len(bs) * epsilon_i-DP by sequential comp
        clipped_sum = adult['Age'].clip(lower=0, upper = b).sum()
        dp_sum = laplace_mech(clipped_sum, sensitivity=b, epsilon=epsilon_i)
        if dp_sum < last_sum:
            return b
        else:
            last_sum = dp_sum
        # dp_sum=laplace_mech(clipped_sum, sensitivity=b, epsilon=0.1)
        # print(f"fDP result for {b} is {dp_sum}")
    
    # raise Exception("No good val!")
    return bs[-1]

pick_b(1.0)
result = laplace_mech(adult['Age'].clip(lower=0, upper= b), sensitivity=b, epsilon=2.0)

In [21]:
# TEST CASE for question 3

many_trials = np.mean([pick_b(1.0) for _ in range(100)])
assert many_trials > 70
assert many_trials < 500

## Question 4

What is the privacy cost of your algorithm, and why?

The privacy cost of pick_b is epsilon because 'len(bs) * epsilon_i = epsilon'.

For each iteration of the loop uses epsilon_i privacy cost which sums to epsilon.

## Question 5

Write code to generate a *histogram* of education numbers in the `adult` dataset.

In [22]:
def education_hist():
    hist = adult['Education'].value_counts()
    return hist


education_hist()

Education
HS-grad         10501
Some-college     7291
Bachelors        5355
Masters          1723
Assoc-voc        1382
11th             1175
Assoc-acdm       1067
10th              933
7th-8th           646
Prof-school       576
9th               514
12th              433
Doctorate         413
5th-6th           333
1st-4th           168
Preschool          51
Name: count, dtype: int64

In [23]:
# TEST CASE for question 5
h = education_hist()
assert h['HS-grad'] == 10501
assert h['12th'] == 433
assert h['Doctorate'] == 413

## Question 6

Write code to release a *differentially private histogram* of education numbers.

In [24]:
def dp_education_hist(epsilon):
    hist = adult['Education'].value_counts()
    noisy_hist = hist.apply(lambda x: laplace_mech(x, sensitivity=1, epsilon=epsilon))
    return noisy_hist
    # total cost is epsilon by parallel comp
    # The elements of the histrogram come from disjoint subsets of the data

dp_education_hist(1.0)

Education
HS-grad         10501.504625
Some-college     7291.328255
Bachelors        5356.446879
Masters          1723.497604
Assoc-voc        1382.457131
11th             1177.671164
Assoc-acdm       1065.098325
10th              932.507552
7th-8th           646.779441
Prof-school       576.178116
9th               515.638859
12th              432.143900
Doctorate         414.496439
5th-6th           334.905913
1st-4th           169.522743
Preschool          50.476405
Name: count, dtype: float64

In [25]:
# TEST CASE for question 6
h = dp_education_hist(1.0)
assert abs(h['HS-grad'] - 10501) < 100
assert abs(h['Doctorate'] - 413) < 100

## Question 7

What is the total privacy cost of `dp_education_hist`, and why?

Sensitivity: Sensitivity of 1 is used in the countng query

Composition: Epsilon by parallel composition

post proc: All elements returned satisfiy DP

## Question 8

Write code to generate a differentially private *contingency table* for the Education and Sex columns of the `adult` dataset.

In [ ]:
def dp_crosstab_education_sex(epsilon):
    # Note: Question asks to group by education then sex, but the test cases need grouping by sex then education
    hist = adult.groupby(['Sex', 'Education']).size()
    # This histogram groups by a combinations of values
    # Each row has exactly one combination of age and sex values
    # (Parallel composition there fore applies)
    return hist.apply(lambda x: laplace_mech(x, sensitivity=1, epsilon=epsilon))
    # If using df, same noise is used for every noise, hist does not have this issue
    # Total privacy cost is epsilon, by parallel compositon

dp_crosstab_education_sex(1.0)

Sex     Education   
Female  10th             295.364066
        11th             431.915839
        12th             144.890524
        1st-4th           47.426108
        5th-6th           83.953257
        7th-8th          160.722695
        9th              142.956393
        Assoc-acdm       420.812362
        Assoc-voc        501.546882
        Bachelors       1619.089286
        Doctorate         87.542269
        HS-grad         3389.925936
        Masters          536.362651
        Preschool         16.160575
        Prof-school       91.031667
        Some-college    2805.984941
Male    10th             638.616650
        11th             743.472345
        12th             290.150481
        1st-4th          123.747878
        5th-6th          249.272063
        7th-8th          485.454319
        9th              369.982559
        Assoc-acdm       646.126291
        Assoc-voc        881.233624
        Bachelors       3735.373116
        Doctorate        326.391363
       

In [32]:
# TEST CASE for question 8
ct = dp_crosstab_education_sex(1.0)
assert abs(ct['Female']['10th'] - 295) < 100
assert abs(ct['Male']['10th'] - 638) < 100
assert abs(ct['Female']['Bachelors'] - 1619) < 100

assert abs(ct['Female']['10th'] - 295) > 0
assert abs(ct['Male']['10th'] - 638) > 0
assert abs(ct['Female']['Bachelors'] - 1619) > 0

## Question 9

Does parallel composition apply for the contingency table in question 1? Why or why not?

Yes we can! The chunks returned by the groupby to create the hist are disjoint, groupby puts rows into groups based on combinations of attribute values, each row has exactly one combination of attribute values.

## Question 10

Does the number of columns used in constructing the contingency table matter for privacy cost? Does it matter for accuracy?

For privacy cost it does not matter. You can use parallel comp for an n-dimensional histogram, for any value of n.
For utility, higher dimensional histograms have much smaller groups, therefore much less signal in each histogram bin. This means you get much lower utility in general because the noise overwhelms the weak signal